# Week 4 — Nemotron Nano 30B Experiments (FDS)
**Model:** `nemotron-3-nano:30b-a3b-q4_K_M` via Ollama (FDS — RTX 5000 Ada 32 GB)  
**Dataset:** `data/personas_sample_500.csv` (same 500-row balanced test set as Week 3)  
**Experiments:**
- Zero-shot classification (no examples)
- Few-shot classification (5 labeled examples)
- Reasoning ON vs Reasoning OFF (unique to Nemotron)
- K-Means cluster analysis

**Baselines to beat (from Week 1):**
- Random Forest: Accuracy 83.70% · Macro F1 0.8366 · AUC-ROC 0.9198
- XGBoost:       Accuracy 76.01% · Macro F1 0.7598 · AUC-ROC 0.8307

**Week 3 results (Nano 4B):**
- Zero-shot: 57.80% · Macro F1 0.5672
- Few-shot:  77.02% · Macro F1 0.7697

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import re
import os
import json

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder, StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

print("Imports OK")

## 2. Configuration

In [ ]:
OLLAMA_URL  = "http://localhost:11434/v1/chat/completions"
MODEL_NAME  = "nemotron-3-nano:30b-a3b-q4_K_M"  # Full 30B model on FDS
SAMPLE_PATH = "../data/personas_sample_500.csv"
RESULTS_DIR = "../results"

# Previous results for comparison
BASELINES = {
    "Random Forest": {"accuracy": 0.8370, "macro_f1": 0.8366, "auc_roc": 0.9198, "ms_per_row": 0.007},
    "XGBoost":       {"accuracy": 0.7601, "macro_f1": 0.7598, "auc_roc": 0.8307, "ms_per_row": 0.001},
    "Nano 4B zero-shot": {"accuracy": 0.5780, "macro_f1": 0.5672, "auc_roc": 0.5821, "ms_per_row": 4728},
    "Nano 4B few-shot":  {"accuracy": 0.7702, "macro_f1": 0.7697, "auc_roc": 0.7695, "ms_per_row": 6503},
}

# Verify Ollama is running
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"Ollama running. Available models: {models}")
    if MODEL_NAME in models:
        print(f"✓ {MODEL_NAME} is ready")
    else:
        print(f"✗ {MODEL_NAME} not found")
        print(f"  Run: ollama pull {MODEL_NAME}")
except Exception as e:
    print(f"✗ Ollama not reachable: {e}")

## 3. Load and Verify Sample

In [ ]:
sample = pd.read_csv(SAMPLE_PATH)

# Filter uninformative occupations
sample = sample[
    ~sample["occupation"].str.lower().str.strip().str.replace("_", " ").isin([
        "no occupation", "not in workforce", ""
    ]) &
    (sample["occupation"].str.strip() != "") &
    (sample["occupation"].notna())
].reset_index(drop=True)

print(f"Sample size: {len(sample)} rows")
print(f"Label balance:")
print(sample["label_name"].value_counts())
print(f"Remaining bad rows: {(sample['occupation'].str.lower().str.strip().str.replace('_', ' ') == 'no occupation').sum()}")
sample.head(3)

## 4. Helper Functions

In [ ]:
SYSTEM_PROMPT = """You are an education level classifier. Given a person's demographic information,
predict whether they are college-educated (have a bachelor's degree or higher) or not.
Think step by step, then respond with ONLY one of these two labels on the final line: college or not_college."""


def serialize_row(row):
    return (
        f"A {int(row['age'])}-year-old {str(row['sex']).lower().strip()}, "
        f"{str(row['marital_status']).replace('_', ' ').strip()}, "
        f"working as a {str(row['occupation']).replace('_', ' ').strip()}. "
        f"Located in {str(row['state']).strip()}."
    )


def build_zero_shot_prompt(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"Classify this person's education level:\n\n"
            f"{serialize_row(row)}\n\n"
            f"Answer with college or not_college only."
        )}
    ]


def build_few_shot_prompt(row, examples):
    example_text = ""
    for ex in examples:
        example_text += f"Person: {serialize_row(ex)}\nLabel: {ex['label_name']}\n\n"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"Here are some labeled examples:\n\n{example_text}"
            f"Now classify this person:\n\n"
            f"{serialize_row(row)}\n\n"
            f"Answer with college or not_college only."
        )}
    ]


def parse_response(content):
    think_match = re.search(r"<think>(.*?)</think>", content, re.DOTALL)
    trace = think_match.group(1).strip() if think_match else ""
    label_match = re.search(r"\b(not_college|college)\b", content, re.IGNORECASE)
    label = label_match.group(1).lower() if label_match else "UNKNOWN"
    return label, trace


def classify_row(messages, enable_thinking=True, timeout=180):
    # Toggle reasoning ON or OFF
    if not enable_thinking:
        messages = [
            {"role": m["role"],
             "content": m["content"] + (" /no_think" if m["role"] == "user" else "")}
            for m in messages
        ]

    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "max_tokens": 2048,
        "temperature": 0.1,
        "stream": False,
        "options": {"num_ctx": 8192},
    }
    start = time.time()
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=timeout).json()
        elapsed = time.time() - start
        content = response["choices"][0]["message"]["content"]
        tokens  = response.get("usage", {}).get("completion_tokens", 0)
        label, trace = parse_response(content)
    except Exception as e:
        elapsed = time.time() - start
        label, trace, tokens, content = "ERROR", str(e), 0, str(e)
    return {
        "label":   label,
        "trace":   trace,
        "raw":     content,
        "time_ms": round(elapsed * 1000),
        "tokens":  tokens,
    }


print("All helper functions defined.")

## 5. Evaluation Helper

In [ ]:
def evaluate_llm_results(results_df, run_name):
    label_map = {"college": 1, "not_college": 0}
    valid = results_df[results_df["pred_label"].isin(["college", "not_college"])].copy()
    unknown_count = len(results_df) - len(valid)

    y_true = valid["true_label"].map(label_map)
    y_pred = valid["pred_label"].map(label_map)

    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average="macro")
    auc  = roc_auc_score(y_true, y_pred)
    avg_ms     = results_df["time_ms"].mean()
    avg_tokens = results_df["tokens"].mean()

    print(f"\n{'='*50}")
    print(f"  {run_name}")
    print(f"{'='*50}")
    print(f"  Rows evaluated:  {len(valid)} / {len(results_df)}")
    print(f"  Unknown:         {unknown_count}")
    print(f"  Accuracy:        {acc:.4f}")
    print(f"  Macro F1:        {f1:.4f}")
    print(f"  AUC-ROC:         {auc:.4f}")
    print(f"  Avg time/row:    {avg_ms:.0f}ms")
    print(f"  Avg tokens:      {avg_tokens:.0f}")
    print()
    print(classification_report(y_true, y_pred, target_names=["not_college", "college"]))

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4, 3))
    ConfusionMatrixDisplay(cm, display_labels=["not_college", "college"]).plot(ax=ax, colorbar=False)
    ax.set_title(run_name)
    plt.tight_layout()
    fname = f"{RESULTS_DIR}/cm_{run_name.replace(' ', '_').lower()}.png"
    plt.savefig(fname, dpi=150)
    plt.show()

    return {
        "model":        run_name,
        "accuracy":     round(acc, 4),
        "macro_f1":     round(f1, 4),
        "auc_roc":      round(auc, 4),
        "ms_per_row":   round(avg_ms, 1),
        "avg_tokens":   round(avg_tokens, 1),
        "unknown":      unknown_count,
        "n_samples":    len(valid),
        "week":         4,
        "device":       "fds-rtx5000",
        "dataset":      "nvidia/Nemotron-Personas-USA",
    }

all_results = []
print("evaluate_llm_results() defined.")

## 6. Few-shot Example Pool

In [ ]:
# Same examples as Week 3 for fair comparison
fs_college = sample[sample["label_name"] == "college"].sample(3, random_state=1)
fs_not     = sample[sample["label_name"] == "not_college"].sample(2, random_state=1)
few_shot_examples = pd.concat([fs_college, fs_not]).sample(frac=1, random_state=1).to_dict("records")

print("Few-shot examples (same as Week 3 for consistency):")
for ex in few_shot_examples:
    print(f"  {serialize_row(ex)} → {ex['label_name']}")

## 7. Experiment A — Zero-shot (500 rows)
> **Expected time:** ~10–15 minutes on FDS (much faster than laptop)

In [ ]:
print(f"Starting zero-shot experiment on {len(sample)} rows...")
print(f"Model: {MODEL_NAME}\n")

zs_results = []
t_start = time.time()

for i, row in sample.iterrows():
    messages = build_zero_shot_prompt(row.to_dict())
    result   = classify_row(messages, enable_thinking=True)

    zs_results.append({
        "row_id":     i,
        "input":      serialize_row(row.to_dict()),
        "true_label": row["label_name"],
        "pred_label": result["label"],
        "correct":    result["label"] == row["label_name"],
        "time_ms":    result["time_ms"],
        "tokens":     result["tokens"],
        "trace":      result["trace"],
        "raw":        result["raw"],
    })

    if (i + 1) % 50 == 0:
        elapsed   = time.time() - t_start
        done      = i + 1
        remaining = (elapsed / done) * (len(sample) - done)
        print(f"  Row {done:3d}/{len(sample)} | Elapsed: {elapsed/60:.1f}min | Remaining: ~{remaining/60:.1f}min")

zs_df = pd.DataFrame(zs_results)
os.makedirs(RESULTS_DIR, exist_ok=True)
zs_df.to_csv(f"{RESULTS_DIR}/week4_zeroshot_raw.csv", index=False)
print(f"\nDone! Total time: {(time.time()-t_start)/60:.1f} minutes")
print(f"Saved: results/week4_zeroshot_raw.csv")

## 8. Zero-shot Results

In [ ]:
zs_metrics = evaluate_llm_results(zs_df, "Nano 30B zero-shot")
all_results.append(zs_metrics)

## 9. Experiment B — Few-shot (500 rows)
> **Expected time:** ~15–20 minutes on FDS

In [ ]:
print(f"Starting few-shot experiment on {len(sample)} rows...")
print(f"Model: {MODEL_NAME}\n")

fs_results = []
t_start = time.time()

for i, row in sample.iterrows():
    messages = build_few_shot_prompt(row.to_dict(), few_shot_examples)
    result   = classify_row(messages, enable_thinking=True)

    fs_results.append({
        "row_id":     i,
        "input":      serialize_row(row.to_dict()),
        "true_label": row["label_name"],
        "pred_label": result["label"],
        "correct":    result["label"] == row["label_name"],
        "time_ms":    result["time_ms"],
        "tokens":     result["tokens"],
        "trace":      result["trace"],
        "raw":        result["raw"],
    })

    if (i + 1) % 50 == 0:
        elapsed   = time.time() - t_start
        done      = i + 1
        remaining = (elapsed / done) * (len(sample) - done)
        print(f"  Row {done:3d}/{len(sample)} | Elapsed: {elapsed/60:.1f}min | Remaining: ~{remaining/60:.1f}min")

fs_df = pd.DataFrame(fs_results)
fs_df.to_csv(f"{RESULTS_DIR}/week4_fewshot_raw.csv", index=False)
print(f"\nDone! Total time: {(time.time()-t_start)/60:.1f} minutes")
print(f"Saved: results/week4_fewshot_raw.csv")

## 10. Few-shot Results

In [ ]:
fs_metrics = evaluate_llm_results(fs_df, "Nano 30B few-shot")
all_results.append(fs_metrics)

## 11. Experiment C — Reasoning ON vs OFF
This is unique to Nemotron — we run the same zero-shot prompt with thinking disabled
and compare accuracy against thinking enabled to measure if reasoning actually helps.

In [ ]:
print(f"Starting reasoning OFF experiment on {len(sample)} rows...")
print(f"Model: {MODEL_NAME} (thinking disabled)\n")

no_think_results = []
t_start = time.time()

for i, row in sample.iterrows():
    messages = build_zero_shot_prompt(row.to_dict())
    result   = classify_row(messages, enable_thinking=False)  # thinking OFF

    no_think_results.append({
        "row_id":     i,
        "input":      serialize_row(row.to_dict()),
        "true_label": row["label_name"],
        "pred_label": result["label"],
        "correct":    result["label"] == row["label_name"],
        "time_ms":    result["time_ms"],
        "tokens":     result["tokens"],
        "trace":      result["trace"],
        "raw":        result["raw"],
    })

    if (i + 1) % 50 == 0:
        elapsed   = time.time() - t_start
        done      = i + 1
        remaining = (elapsed / done) * (len(sample) - done)
        print(f"  Row {done:3d}/{len(sample)} | Elapsed: {elapsed/60:.1f}min | Remaining: ~{remaining/60:.1f}min")

nt_df = pd.DataFrame(no_think_results)
nt_df.to_csv(f"{RESULTS_DIR}/week4_nothink_raw.csv", index=False)
print(f"\nDone! Total time: {(time.time()-t_start)/60:.1f} minutes")
print(f"Saved: results/week4_nothink_raw.csv")

## 12. Reasoning ON vs OFF Results

In [ ]:
nt_metrics = evaluate_llm_results(nt_df, "Nano 30B reasoning OFF")
all_results.append(nt_metrics)

# Direct comparison
print("\n=== REASONING ON vs OFF ===")
print(f"  Reasoning ON  (zero-shot): {zs_metrics['accuracy']:.4f} accuracy | {zs_metrics['macro_f1']:.4f} F1 | {zs_metrics['ms_per_row']:.0f}ms/row")
print(f"  Reasoning OFF (zero-shot): {nt_metrics['accuracy']:.4f} accuracy | {nt_metrics['macro_f1']:.4f} F1 | {nt_metrics['ms_per_row']:.0f}ms/row")
acc_diff = zs_metrics['accuracy'] - nt_metrics['accuracy']
print(f"  Accuracy difference: {acc_diff:+.4f} ({'reasoning helps' if acc_diff > 0 else 'reasoning hurts' if acc_diff < 0 else 'no difference'})")

## 13. Reasoning Traces Analysis
The 30B model should produce reasoning traces — extract the best examples.

In [ ]:
traces_df = zs_df[zs_df["trace"].str.len() > 20].copy()
print(f"Rows with reasoning traces: {len(traces_df)} / {len(zs_df)}")

if len(traces_df) > 0:
    print("\n=== CORRECT predictions with reasoning ===")
    for _, row in traces_df[traces_df["correct"] == True].head(3).iterrows():
        print(f"Input:     {row['input']}")
        print(f"Label:     {row['true_label']}")
        print(f"Reasoning: {row['trace'][:300]}")
        print()

    print("\n=== WRONG predictions with reasoning ===")
    for _, row in traces_df[traces_df["correct"] == False].head(3).iterrows():
        print(f"Input:     {row['input']}")
        print(f"True: {row['true_label']} | Predicted: {row['pred_label']}")
        print(f"Reasoning: {row['trace'][:300]}")
        print()

    # Save best traces for report
    traces_df[["input", "true_label", "pred_label", "correct", "trace"]].to_csv(
        f"{RESULTS_DIR}/week4_reasoning_traces.csv", index=False
    )
    print("Saved: results/week4_reasoning_traces.csv")
else:
    print("No reasoning traces found in outputs.")

## 14. Full Comparison — All Models

In [ ]:
baseline_rows = [
    {"model": "Random Forest",      "accuracy": 0.8370, "macro_f1": 0.8366, "auc_roc": 0.9198, "ms_per_row": 0.007,  "week": 1},
    {"model": "XGBoost",            "accuracy": 0.7601, "macro_f1": 0.7598, "auc_roc": 0.8307, "ms_per_row": 0.001,  "week": 1},
    {"model": "Nano 4B zero-shot",  "accuracy": 0.5780, "macro_f1": 0.5672, "auc_roc": 0.5821, "ms_per_row": 4728.3, "week": 3},
    {"model": "Nano 4B few-shot",   "accuracy": 0.7702, "macro_f1": 0.7697, "auc_roc": 0.7695, "ms_per_row": 6503.7, "week": 3},
]

comparison_df = pd.DataFrame(baseline_rows + all_results)
print("=== FULL COMPARISON: All Models ===")
print(comparison_df[["model", "accuracy", "macro_f1", "auc_roc", "ms_per_row", "week"]].to_string(index=False))

In [ ]:
# Full comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ["accuracy", "macro_f1", "auc_roc"]
titles  = ["Accuracy", "Macro F1", "AUC-ROC"]
colors  = ["#378ADD", "#1D9E75", "#EF9F27", "#D85A30", "#8B5CF6", "#EC4899", "#06B6D4"]

for ax, metric, title in zip(axes, metrics, titles):
    vals  = comparison_df[metric].values
    names = [n.replace(" ", "\n") for n in comparison_df["model"].values]
    bars  = ax.bar(names, vals, color=colors[:len(vals)])
    ax.set_title(title)
    ax.set_ylim(0.4, 1.0)
    ax.tick_params(axis="x", rotation=15, labelsize=8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)

plt.suptitle("Full Comparison: Traditional ML vs Nemotron Nano 4B vs 30B", fontsize=12)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/week4_full_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/week4_full_comparison.png")

## 15. Label Noise Analysis (30B)

In [ ]:
COLLEGE_OCCUPATIONS = [
    "software developer", "engineer", "doctor", "physician", "surgeon",
    "lawyer", "attorney", "architect", "nurse", "registered nurse",
    "accountant", "auditor", "financial manager", "occupational therapist",
    "database administrator", "teacher", "professor", "analyst"
]

potential_noise = zs_df[
    (zs_df["true_label"] == "not_college") &
    (zs_df["pred_label"] == "college") &
    (zs_df["input"].str.lower().str.contains("|".join(COLLEGE_OCCUPATIONS)))
].copy()

print(f"Potential label noise rows (30B model): {len(potential_noise)}")
print(f"Compare to Week 3 (4B model): 23 rows")
print()
for _, row in potential_noise.head(5).iterrows():
    print(f"Input: {row['input']}")
    print(f"True: {row['true_label']} | Predicted: {row['pred_label']}")
    if row['trace']:
        print(f"Reasoning: {row['trace'][:150]}")
    print()

## 16. K-Means Cluster Analysis (30B)

In [ ]:
# Encode features for clustering
cluster_features = ['age', 'sex', 'marital_status', 'occupation', 'state']
X_cluster = sample[cluster_features].copy()

le_dict = {}
for col in ['sex', 'marital_status', 'occupation', 'state']:
    le = LabelEncoder()
    X_cluster[col] = le.fit_transform(X_cluster[col].astype(str))
    le_dict[col] = le

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Use same k=4 as Week 3 for direct comparison
K = 4
km = KMeans(n_clusters=K, random_state=42, n_init=10)
cluster_labels_arr = km.fit_predict(X_scaled)

# Add cluster labels to all results
zs_clustered = zs_df.copy()
fs_clustered = fs_df.copy()
nt_clustered = nt_df.copy()
zs_clustered['cluster'] = cluster_labels_arr
fs_clustered['cluster'] = cluster_labels_arr
nt_clustered['cluster'] = cluster_labels_arr

cluster_stats = []

for c in range(K):
    zs_rows = zs_clustered[zs_clustered['cluster'] == c]
    fs_rows = fs_clustered[fs_clustered['cluster'] == c]
    nt_rows = nt_clustered[nt_clustered['cluster'] == c]

    zs_acc = zs_rows['correct'].mean()
    fs_acc = fs_rows['correct'].mean()
    nt_acc = nt_rows['correct'].mean()
    size   = len(zs_rows)
    college_rate = (zs_rows['true_label'] == 'college').mean()

    cluster_row_ids = zs_rows['row_id'].values
    top_occ = sample.loc[sample.index.isin(cluster_row_ids), 'occupation'].value_counts().head(3).index.tolist()
    avg_age = sample.loc[sample.index.isin(cluster_row_ids), 'age'].mean()

    cluster_stats.append({
        'cluster':         c,
        'size':            size,
        'zs_accuracy':     round(zs_acc, 3),
        'fs_accuracy':     round(fs_acc, 3),
        'nt_accuracy':     round(nt_acc, 3),
        'college_rate':    round(college_rate, 3),
        'avg_age':         round(avg_age, 1),
        'top_occupations': ', '.join(top_occ),
    })

    print(f'\n── Cluster {c} ──────────────────────────────────────')
    print(f'  Size:                {size} rows')
    print(f'  Zero-shot accuracy:  {zs_acc:.1%}')
    print(f'  Few-shot accuracy:   {fs_acc:.1%}')
    print(f'  No-think accuracy:   {nt_acc:.1%}')
    print(f'  College rate:        {college_rate:.1%}')
    print(f'  Avg age:             {avg_age:.1f}')
    print(f'  Top occupations:     {", ".join(top_occ)}')

cluster_df = pd.DataFrame(cluster_stats)
cluster_df.to_csv(f'{RESULTS_DIR}/week4_cluster_analysis.csv', index=False)
print('\nSaved: results/week4_cluster_analysis.csv')

In [ ]:
# Cluster visualization — 4B vs 30B comparison
# Load Week 3 cluster data for comparison
try:
    w3_cluster = pd.read_csv(f'{RESULTS_DIR}/week3_cluster_analysis.csv')
    has_w3 = True
except:
    has_w3 = False

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

x = range(K)
width = 0.35

# Zero-shot: 4B vs 30B
if has_w3:
    axes[0].bar([i - width/2 for i in x], w3_cluster['zs_accuracy'], width,
                label='Nano 4B', color='#378ADD', alpha=0.8)
axes[0].bar([i + width/2 for i in x] if has_w3 else x,
            cluster_df['zs_accuracy'], width if has_w3 else 0.6,
            label='Nano 30B', color='#1D9E75', alpha=0.8)
axes[0].axhline(y=zs_df['correct'].mean(), color='red', linestyle='--',
                label=f'30B overall ({zs_df["correct"].mean():.1%})')
axes[0].set_title('Zero-shot Accuracy by Cluster')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1)
axes[0].set_xticks(list(x))
axes[0].legend(fontsize=8)

# Few-shot accuracy
sns.barplot(data=cluster_df, x='cluster', y='fs_accuracy',
            color='#1D9E75', ax=axes[1])
axes[1].axhline(y=fs_df['correct'].mean(), color='red', linestyle='--',
                label=f'Overall ({fs_df["correct"].mean():.1%})')
axes[1].set_title('Few-shot Accuracy by Cluster (30B)')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=8)

# Reasoning ON vs OFF
axes[2].bar([i - width/2 for i in x], cluster_df['zs_accuracy'], width,
            label='Reasoning ON', color='#8B5CF6', alpha=0.8)
axes[2].bar([i + width/2 for i in x], cluster_df['nt_accuracy'], width,
            label='Reasoning OFF', color='#EC4899', alpha=0.8)
axes[2].set_title('Reasoning ON vs OFF by Cluster')
axes[2].set_xlabel('Cluster')
axes[2].set_ylabel('Accuracy')
axes[2].set_ylim(0, 1)
axes[2].set_xticks(list(x))
axes[2].legend(fontsize=8)

plt.suptitle(f'K-Means Cluster Analysis (k={K}) — Nano 30B vs 4B', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/week4_cluster_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/week4_cluster_accuracy.png')

## 17. Save All Results to metrics.csv

In [ ]:
metrics_path = f"{RESULTS_DIR}/metrics.csv"

try:
    existing = pd.read_csv(metrics_path)
    existing = existing[existing["week"] != 4]
    week4_df = pd.DataFrame(all_results)
    updated  = pd.concat([existing, week4_df], ignore_index=True)
except FileNotFoundError:
    updated = pd.DataFrame(all_results)

updated.to_csv(metrics_path, index=False)
print(f"Saved: {metrics_path}")
print()
print(updated[["model", "accuracy", "macro_f1", "auc_roc", "ms_per_row", "week"]].to_string(index=False))

## 18. Week 4 Summary

Fill in after running all cells:

| Model | Accuracy | Macro F1 | AUC-ROC | Time/row |
|---|---|---|---|---|
| Random Forest (Week 1) | 83.70% | 0.8366 | 0.9198 | 0.007 ms |
| XGBoost (Week 1) | 76.01% | 0.7598 | 0.8307 | 0.001 ms |
| Nano 4B zero-shot (Week 3) | 57.80% | 0.5672 | 0.5821 | 4,728 ms |
| Nano 4B few-shot (Week 3) | 77.02% | 0.7697 | 0.7695 | 6,503 ms |
| Nano 30B zero-shot | ___ | ___ | ___ | ___ ms |
| Nano 30B few-shot | ___ | ___ | ___ | ___ ms |
| Nano 30B reasoning OFF | ___ | ___ | ___ | ___ ms |

**Observations:**
- Did 30B outperform 4B?
- Did reasoning ON help vs reasoning OFF?
- Were reasoning traces produced this time?
- Which cluster benefited most from the 30B model?

**Commit before moving on:**
```
git add .
git commit -m "Week 4: Nano 30B zero-shot, few-shot, reasoning ON/OFF, K-Means"
git push
```

**Next week:** QLoRA fine-tuning on FDS + error analysis + feature attribution comparison.